# GraphEm: from a graph to a radial ranking

This notebook is a compact tour of the current `graphem-jax` 0.2 API. It builds a small graph, refines a three-dimensional layout, ranks every vertex by its distance from the origin, and measures full-population Spearman correlations with several familiar centralities.

The radial score is an empirical proxy. It is not an exact centrality measure, and a favorable correlation on one graph family does not establish a universal law. Keep the graph, parameters, seed, and node order with every result.

## Installation

Run the notebook from the repository root after installing the checkout:

```bash
python -m pip install -e .
```

For a released installation, use the command in the project README. Installation is deliberately not performed from a notebook cell: cloning into the current directory or silently upgrading packages makes a rerun depend on hidden state.

In [ ]:
import json
import platform

import jax
import networkx as nx
import numpy as np
import scipy
from scipy.stats import spearmanr

import graphem as ge

print(
    {
        "python": platform.python_version(),
        "graphem": ge.__version__,
        "jax": jax.__version__,
        "numpy": np.__version__,
        "scipy": scipy.__version__,
    }
)

## Build one explicit graph

GraphEm accepts a square dense array or SciPy sparse adjacency object. The bundled graph generators return SciPy sparse adjacency objects with contiguous integer node labels. Here every source of variation is named rather than left implicit.

In [ ]:
GRAPH_SEED = 7
GRAPH_PARAMETERS = {"n": 96, "p": 0.08, "seed": GRAPH_SEED}

adjacency = ge.generate_er(**GRAPH_PARAMETERS)
graph = nx.from_scipy_sparse_array(adjacency)

assert adjacency.shape == (GRAPH_PARAMETERS["n"], GRAPH_PARAMETERS["n"])
assert adjacency.nnz % 2 == 0
assert list(graph.nodes()) == list(range(GRAPH_PARAMETERS["n"]))

print(
    f"Erdős–Rényi graph: {graph.number_of_nodes()} vertices, "
    f"{graph.number_of_edges()} undirected edges"
)

## Refine the layout

The constructor below uses the public adjacency-object API. `sample_size` controls how many edge midpoints are queried per iteration; `batch_size` bounds the tiled nearest-neighbour calculation. Both are capped internally when the graph is smaller than the request.

These values make the example quick on CPU. They are demonstration settings, not tuned benchmark parameters.

In [ ]:
LAYOUT_PARAMETERS = {
    "n_components": 3,
    "L_min": 1.0,
    "k_attr": 0.2,
    "k_inter": 0.5,
    "n_neighbors": 8,
    "sample_size": 64,
    "batch_size": 64,
    "seed": GRAPH_SEED,
    "verbose": False,
}
LAYOUT_ITERATIONS = 8

# SciPy's sparse eigensolver may choose an initial vector through NumPy.
# Seeding NumPy makes that otherwise implicit input visible in this example.
np.random.seed(GRAPH_SEED)
embedder = ge.GraphEmbedder(adjacency=adjacency, **LAYOUT_PARAMETERS)
positions = embedder.run_layout(num_iterations=LAYOUT_ITERATIONS)

assert positions.shape == (GRAPH_PARAMETERS["n"], LAYOUT_PARAMETERS["n_components"])
assert np.isfinite(np.asarray(positions)).all()
print("layout shape:", positions.shape)

## Rank vertices by radius

The score is the Euclidean distance from the layout origin. We sort by decreasing radius and use the integer node ID as a stable tie-breaker. Larger radius means earlier rank; it does not, by itself, mean greater influence under a diffusion model.

In [ ]:
positions_host = np.asarray(positions)
radii = np.linalg.norm(positions_host, axis=1)
node_ids = np.arange(radii.size, dtype=np.int64)
ranked_nodes = np.lexsort((node_ids, -radii))

top_ten = [
    {"node": int(node), "radius": float(radii[node])}
    for node in ranked_nodes[:10]
]
top_ten

## Measure full-ranking agreement

Spearman's rho compares complete rankings and handles tied centrality values with average ranks. A value near `1` means similar ordering, a value near `-1` means reversed ordering, and a value near `0` means little monotone agreement. The calculation below uses every vertex in the shared node order.

In [ ]:
degree = np.asarray([graph.degree(int(node)) for node in node_ids], dtype=np.float64)
pagerank_by_node = nx.pagerank(graph)
pagerank = np.asarray([pagerank_by_node[int(node)] for node in node_ids])
core_by_node = nx.core_number(graph)
k_core = np.asarray([core_by_node[int(node)] for node in node_ids], dtype=np.float64)

centralities = {
    "degree": degree,
    "pagerank": pagerank,
    "k_core": k_core,
}
spearman_rho = {
    name: float(spearmanr(radii, values).statistic)
    for name, values in centralities.items()
}

for name, rho in spearman_rho.items():
    print(f"radius versus {name:8s}: Spearman rho = {rho: .4f}")

## Keep the run record beside the numbers

A score without its graph and parameter record is difficult to reproduce and easy to over-interpret. This lightweight record is suitable for an exploratory notebook. Publication evidence should additionally bind exact graph hashes, source and environment receipts, raw arrays, and an independent audit.

In [ ]:
run_record = {
    "package": {"name": "graphem-jax", "version": ge.__version__},
    "graph": {
        "generator": "graphem.generate_er",
        "parameters": GRAPH_PARAMETERS,
        "vertices": graph.number_of_nodes(),
        "undirected_edges": graph.number_of_edges(),
        "node_order": "contiguous integer IDs, ascending",
    },
    "layout": {**LAYOUT_PARAMETERS, "num_iterations": LAYOUT_ITERATIONS},
    "score": "Euclidean radius after layout; decreasing order; node-ID tie-break",
    "quality": {
        "population": graph.number_of_nodes(),
        "metric": "Spearman rank correlation with average ties",
        "rho": spearman_rho,
    },
}

print(json.dumps(run_record, indent=2, sort_keys=True))

## Try another graph family without changing the protocol silently

The public generators all return the same adjacency-object shape, but graph families can behave very differently. Change one declared graph specification at a time and retain the old record rather than overwriting it.

In [ ]:
graph_families = {
    "small world": lambda: ge.generate_ws(n=96, k=6, p=0.2, seed=GRAPH_SEED),
    "preferential attachment": lambda: ge.generate_ba(n=96, m=3, seed=GRAPH_SEED),
    "grid": lambda: ge.generate_road_network(width=12, height=8),
    "balanced tree": lambda: ge.generate_balanced_tree(r=2, h=6),
}

for name, make_adjacency in graph_families.items():
    candidate = make_adjacency()
    print(name, candidate.shape[0], candidate.nnz // 2)

## Inspect the real-world dataset catalogue

Listing the catalogue is local and does not download anything. Loading a dataset may fetch an external archive. For reproducible work, record its landing page, archive and decompressed hashes, parser, component rule, relabelling rule, and final graph identity.

In [ ]:
from graphem.datasets import list_available_datasets

dataset_catalogue = list_available_datasets()
for dataset_id in sorted(dataset_catalogue)[:8]:
    item = dataset_catalogue[dataset_id]
    print(f"{dataset_id:28s} {item['source']:20s} {item['description']}")

## Next steps

- Use `graphem.benchmark.benchmark_correlations` for a small convenience benchmark.
- Use `graphem.graphem_seed_selection` only as a heuristic, and compare influence methods on shared propagation worlds or sufficiently large independent samples.
- Use the repository's audited reproduction harness for scientific benchmark claims.

The production CUDA implementation and large-graph tooling live in `graphem-rapids`; this notebook intentionally stays within the public API of this repository.